# Proyecto Final - PySpark (Databricks)
Práctica integradora de 20 puntos sobre un DataFrame de personas (nombre, edad, ciudad).

## Paso adicional: imports y SparkSession
En Databricks la sesión `spark` ya existe por defecto; `getOrCreate()` la reutiliza en vez de crear una nueva.

In [ ]:
# Importamos SparkSession para crear/obtener la sesión de Spark
from pyspark.sql import SparkSession
# Importamos las funciones de columna que vamos a usar en todo el notebook
from pyspark.sql.functions import col, avg, sum as spark_sum, min as spark_min, max as spark_max, when, concat, lit

# getOrCreate() reutiliza la sesión "spark" que Databricks ya deja creada,
# en vez de crear una nueva (evita conflictos de recursos)
spark = SparkSession.builder.appName("ProyectoFinalPySpark").getOrCreate()

## 1. Crear un DataFrame con datos de personas (nombre, edad, ciudad)

In [ ]:
# Lista de tuplas: cada tupla es una fila (Nombre, Edad, Ciudad)
data = [("Alice", 25, "New York"),
        ("Bob", 30, "Los Angeles"),
        ("Charlie", 22, "Chicago")]
# Nombres de columnas, en el mismo orden que los valores de cada tupla
columns = ["Nombre", "Edad", "Ciudad"]

# Creamos el DataFrame a partir de los datos y el esquema de columnas
df = spark.createDataFrame(data, columns)
df.show()

## 2. Mostrar solo los nombres de las personas

In [ ]:
# select() elige solo la(s) columna(s) indicadas, descarta el resto
df.select("Nombre").show()

## 3. Filtrar personas cuya edad sea mayor o igual a 25

In [ ]:
# filter() se queda solo con las filas donde la condición es verdadera (>= 25)
df.filter(col("Edad") >= 25).show()

## 4. Agregar una nueva columna "Pais" con un valor constante

In [ ]:
# withColumn() agrega una columna nueva; lit() indica que el valor es constante
# (el mismo para todas las filas), no una columna existente
df_pais = df.withColumn("Pais", lit("Estados Unidos"))
df_pais.show()

## 5. Calcular el promedio de edad de todas las personas

In [ ]:
# avg() es una función de agregación: calcula el promedio sobre toda la columna
# alias() le da un nombre legible a la columna resultante
df.select(avg("Edad").alias("EdadPromedio")).show()

## 6. Ordenar el DataFrame por edad en orden descendente

In [ ]:
# orderBy() ordena las filas; desc() invierte el orden a descendente
df.orderBy(col("Edad").desc()).show()

## 7. Agrupar por ciudad y calcular la cantidad de personas en cada ciudad

In [ ]:
# groupBy() agrupa las filas que comparten el mismo valor de "Ciudad",
# count() cuenta cuántas filas cayeron en cada grupo
df.groupBy("Ciudad").count().show()

## 8. Renombrar la columna "Nombre" a "NombreCompleto"

In [ ]:
# withColumnRenamed(nombre_actual, nombre_nuevo) cambia el nombre de una columna
# sin tocar sus valores
df_renombrado = df.withColumnRenamed("Nombre", "NombreCompleto")
df_renombrado.show()

## 9. Eliminar la columna "Edad" del DataFrame

In [ ]:
# drop() quita una columna del DataFrame; devuelve un DataFrame nuevo
# (df original queda intacto, por eso se guarda en df_sin_edad)
df_sin_edad = df.drop("Edad")
df_sin_edad.show()

## 10. Consulta SQL sobre el DataFrame: personas mayores de 20 años

In [ ]:
# createOrReplaceTempView() registra el DataFrame como una "tabla" temporal
# llamada "personas", para poder consultarla con SQL estándar
df.createOrReplaceTempView("personas")
spark.sql("SELECT * FROM personas WHERE Edad > 20").show()

## 11. Calcular la suma total de todas las edades

In [ ]:
# sum() (importado como spark_sum para no chocar con la función nativa de Python)
# suma todos los valores de la columna "Edad"
df.select(spark_sum("Edad").alias("SumaEdades")).show()

## 12. Calcular la edad mínima y máxima de todas las personas

In [ ]:
# min() y max() (renombrados igual que sum, por el mismo motivo) devuelven
# el valor mínimo y máximo de la columna en una sola fila
df.select(spark_min("Edad").alias("EdadMinima"), spark_max("Edad").alias("EdadMaxima")).show()

## 13. Filtrar personas de "Chicago" con edad menor a 30

In [ ]:
# Se pueden combinar varias condiciones con & (AND) u | (OR);
# cada condición individual va entre paréntesis
df.filter((col("Ciudad") == "Chicago") & (col("Edad") < 30)).show()

## 14. Agregar una nueva columna "EdadDuplicada" con el doble de la edad

In [ ]:
# col("Edad") * 2 opera sobre toda la columna a la vez (vectorizado),
# no fila por fila en Python como haría una UDF
df_edad_duplicada = df.withColumn("EdadDuplicada", col("Edad") * 2)
df_edad_duplicada.show()

## 15. Convertir todas las edades en años a meses

In [ ]:
# 1 año = 12 meses, por eso multiplicamos la columna "Edad" por 12
df_meses = df.withColumn("EdadEnMeses", col("Edad") * 12)
df_meses.show()

## 16. Contar el número total de personas en el DataFrame

In [ ]:
# count() sobre el DataFrame (sin agrupar) cuenta el total de filas
total_personas = df.count()
print(f"Total de personas: {total_personas}")

## 17. Filtrar personas cuya edad sea un número par

In [ ]:
# % es el operador módulo (resto de la división); edad par significa resto 0 al dividir entre 2
df.filter(col("Edad") % 2 == 0).show()

## 18. Cantidad de personas por rango de edades (0-20, 21-40, 41-60, 61+)

In [ ]:
# when().when()...otherwise() es como un if/elif/else pero para columnas:
# evalúa cada condición en orden y usa la primera que se cumpla
df_rangos = df.withColumn(
    "RangoEdad",
    when(col("Edad") <= 20, "0-20")
    .when(col("Edad") <= 40, "21-40")
    .when(col("Edad") <= 60, "41-60")
    .otherwise("61+")
)
# Con la columna de rango ya creada, agrupamos y contamos por rango
df_rangos.groupBy("RangoEdad").count().show()

## 19. Contar cuántas personas tienen el mismo nombre

In [ ]:
# Agrupamos por "Nombre": si hubiera nombres repetidos, count() los sumaría en un solo grupo
# (con estos 3 nombres únicos, cada grupo da 1)
df.groupBy("Nombre").count().withColumnRenamed("count", "Repeticiones").show()

## 20. Concatenar "Nombre" y "Ciudad" en una nueva columna "InformacionPersonal"

In [ ]:
# concat() une varias columnas/valores en una sola columna de texto;
# lit(" - ") agrega un separador literal entre nombre y ciudad
df_info = df.withColumn("InformacionPersonal", concat(col("Nombre"), lit(" - "), col("Ciudad")))
df_info.show()